In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import stim
from itertools import combinations

In [ ]:
def build_golay24_H() -> np.ndarray:
    """Build H = [B | I_12] for the extended binary Golay code [24,12,8]."""
    B = np.zeros((12, 12), dtype=np.uint8)
    qr_11 = {1, 3, 4, 5, 9}

    # First row/col ones (except diagonal handled later)
    for i in range(1, 12):
        B[0, i] = 1
        B[i, 0] = 1

    # Quadratic residue rule for indices 1..11
    for i in range(1, 12):
        for j in range(1, 12):
            if i == j:
                B[i, j] = 1
            elif (j - i) % 11 in qr_11:
                B[i, j] = 1

    H = np.hstack([B, np.eye(12, dtype=np.uint8)]).astype(np.uint8)
    return H


def _pack_bits_to_int(bits: np.ndarray) -> int:
    """Pack a length-m array of 0/1 bits into an integer using little-endian order."""
    # bits[0] is LSB
    m = bits.size
    out = 0
    for i in range(m):
        out |= (int(bits[i]) & 1) << i
    return out


def build_syndrome_correction_table_golay24(H: np.ndarray) -> np.ndarray:
    """
    Build a complete correction lookup table for extended Golay [24,12,8].

    Returns:
      corr_table: np.uint32 array of shape (4096,)
        corr_table[s] is a 24-bit mask to XOR with the measured data bits.

    Strategy:
      Fill syndromes with minimal-weight coset leaders by enumerating error masks
      in increasing weight order up to weight 4 (covering radius 4).
    """
    H = (H.copy() & 1).astype(np.uint8)
    m, n = H.shape  # m=12, n=24
    assert m == 12 and n == 24

    # Pack each column of H into a 12-bit integer
    col_int = np.zeros(n, dtype=np.uint16)
    for j in range(n):
        col_int[j] = _pack_bits_to_int(H[:, j])

    # Correction table indexed by 12-bit syndrome (0..4095)
    corr_table = np.zeros(1 << m, dtype=np.uint32)
    filled = np.zeros(1 << m, dtype=bool)

    # Weight 0
    filled[0] = True
    corr_table[0] = 0

    # Helper to set entry if empty
    def try_set(syn: int, mask: int):
        if not filled[syn]:
            filled[syn] = True
            corr_table[syn] = np.uint32(mask)

    # Weight 1
    for i in range(n):
        syn = int(col_int[i])
        mask = 1 << i
        try_set(syn, mask)

    # Weight 2
    for i, j in combinations(range(n), 2):
        syn = int(col_int[i] ^ col_int[j])
        mask = (1 << i) ^ (1 << j)
        try_set(syn, mask)

    # Weight 3
    for i, j, k in combinations(range(n), 3):
        syn = int(col_int[i] ^ col_int[j] ^ col_int[k])
        mask = (1 << i) ^ (1 << j) ^ (1 << k)
        try_set(syn, mask)

    # Weight 4 (covering radius for extended Golay)
    for i, j, k, l in combinations(range(n), 4):
        syn = int(col_int[i] ^ col_int[j] ^ col_int[k] ^ col_int[l])
        mask = (1 << i) ^ (1 << j) ^ (1 << k) ^ (1 << l)
        try_set(syn, mask)

    if not np.all(filled):
        missing = np.where(~filled)[0]
        raise RuntimeError(f"Correction table incomplete, missing {missing.size} syndromes.")

    return corr_table


H = build_golay24_H()
corr_table = build_syndrome_correction_table_golay24(H)

print("H shape:", H.shape)
print("HH^T mod 2 == 0 ?", bool(np.all((H @ H.T) % 2 == 0)))
print("Correction table size:", corr_table.size, "(expected 4096)")

In [ ]:
def golay24_x_memory_circuit(H: np.ndarray, p: float) -> stim.Circuit:
    """
    One-shot X-error memory with Z-parity checks defined by H.

    Qubits:
      data: 0..23
      anc:  24..35  (12 ancillas, one per check row)

    Noise:
      X_ERROR(p) on each data qubit (independent bit-flips).

    Measurements (in order):
      12 ancilla Z measurements (syndrome bits)
      24 data Z measurements (data bits after noise, before correction)
    """
    H = (H.copy() & 1).astype(np.uint8)
    m, n = H.shape
    assert m == 12 and n == 24

    data = list(range(n))
    anc = list(range(n, n + m))  # 24..35

    c = stim.Circuit()

    # Prepare |0...0> for data and ancillas
    c.append("R", data + anc)

    # Apply bit-flip noise to data qubits
    c.append("X_ERROR", data, p)

    # Parity checks: ancilla accumulates XOR of selected data bits
    # Implemented with CX from data -> anc
    for row in range(m):
        ops = []
        a = anc[row]
        ones = np.where(H[row] == 1)[0]
        for q in ones:
            ops += [int(q), int(a)]
        if ops:
            c.append("CX", ops)

    # Measure ancillas (syndrome)
    c.append("M", anc)

    # Measure data (so decoding success can be evaluated)
    c.append("M", data)

    return c

In [ ]:
def simulate_golay24_quantum(H: np.ndarray, corr_table: np.ndarray, p: float, shots: int, seed: int = 0) -> float:
    """
    Quantum simulation using Stim:
      - sample syndrome bits from ancilla measurements
      - sample data bits from data measurements
      - decode with corr_table (syndrome -> correction mask)
      - count block failure if residual != 0

    Returns:
      block_error_rate
    """
    H = (H.copy() & 1).astype(np.uint8)
    m, n = H.shape
    assert m == 12 and n == 24
    assert corr_table.size == 4096

    c = golay24_x_memory_circuit(H, p)
    sampler = c.compile_sampler(seed=seed)
    ms = sampler.sample(shots)  # shape: (shots, 12 + 24)

    synd = ms[:, :m].astype(np.uint16)
    data = ms[:, m:].astype(np.uint32)

    # Vectorized packing of bits into ints (little-endian)
    w12 = (1 << np.arange(m, dtype=np.uint16))
    w24 = (1 << np.arange(n, dtype=np.uint32))

    syn_int = (synd @ w12).astype(np.uint16)     # (shots,)
    data_int = (data @ w24).astype(np.uint32)    # (shots,)

    corr_int = corr_table[syn_int]               # (shots,)
    residual = data_int ^ corr_int

    ber = float(np.mean(residual != 0))
    return ber


# --- Parameters ---
p_values = [0.001, 0.005, 0.01, 0.02, 0.05, 0.08, 0.10, 0.12, 0.15]
shots = 1_000_000

# --- Print an example circuit (p=0.01) ---
c_demo = golay24_x_memory_circuit(H, p=0.01)
print(c_demo)

# --- Run simulation ---
bers = []
for p in p_values:
    ber = simulate_golay24_quantum(H, corr_table, p, shots, seed=12345)
    bers.append(ber)
    print(f"p={p:.3f} -> block error rate = {ber:.6e}")

# --- Theoretical curve for extended Golay [24,12,8] under BSC(p) with guaranteed unique decoding up to t=3 ---
# Failure occurs when >= 4 bit flips happen (because d=8 -> t=floor((d-1)/2)=3)
import math

theory = []
for p in p_values:
    ok = 0.0
    for k in range(0, 4):
        ok += math.comb(24, k) * (p**k) * ((1 - p)**(24 - k))
    theory.append(1.0 - ok)

# --- Plot ---
plt.figure(figsize=(8, 5))
plt.semilogy(p_values, [max(x, 1e-12) for x in bers], 'o-', label='Sim (Stim) + syndrome decoding')
plt.semilogy(p_values, [max(x, 1e-12) for x in theory], 's-', label='Theory Golay [24,12,8] (t=3): P(W ≥ 4)')
plt.semilogy(p_values, p_values, '--', label='Physical p (no correction)')
plt.xlabel('Physical bit-flip probability p')
plt.ylabel('Block error rate')
plt.title('Extended Golay [24,12,8] – quantum (Stim) vs theoretical bound')
plt.grid(True, which='both', alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 1. Genera la matrice H (se non è già caricata in memoria)
# Crea il circuito quantistico di Stim con un rumore p = 0.01 usando la matrice H già esistente
p_demo = 0.01
circuito_demo = golay24_x_memory_circuit(H, p_demo)

# Disegna il circuito in formato grafico (SVG)
circuito_demo.diagram('timeline-svg')

# 2. Crea il circuito quantistico di Stim con un rumore p = 0.01
p_demo = 0.01
circuito_demo = golay24_x_memory_circuit(H, p_demo)

# 3. Stampa tutte le istruzioni testuali del circuito
print("--- ISTRUZIONI STIM DEL CIRCUITO ---")
print(repr(circuito_demo))

# 4. Disegna il circuito in formato grafico (SVG)
circuito_demo.diagram('timeline-svg')

In [ ]:
# Golay(24,12,8): p_L vs p using hierarchical Monte Carlo concatenation (L=1,2,3)
# L=1 uses the Stim circuit.
# L=2 and L=3 are built by sampling 24 outputs from the previous level and decoding with the outer Golay decoder.

import numpy as np
import matplotlib.pyplot as plt

# -------------------------
# Configuration
# -------------------------
levels = [1, 2, 3]
eps = 1e-12

# Physical error rates (log-spaced, where Golay helps)
p_min, p_max = 1e-4, 8e-2
num_points = 24
p_values = np.geomspace(p_min, p_max, num_points)

# Shots per level (keep L2/L3 smaller; L1 is the expensive Stim part)
shots_L1 = 300_000
shots_L2 = 200_000
shots_L3 = 100_000

seed_base = 12345

# -------------------------
# Helpers
# -------------------------
def sample_level1_fail_bits_from_stim(H: np.ndarray, corr_table: np.ndarray, p: float, shots: int, seed: int) -> np.ndarray:
    """
    Run the Stim circuit for Golay(24,12,8) X-memory and return a 0/1 array:
      fail=1 if decoding does NOT return the all-zero codeword (i.e., corrected data != 0).
    """
    H = (H.copy() & 1).astype(np.uint8)
    m, n = H.shape
    assert (m, n) == (12, 24)
    assert corr_table.size == 4096

    c = golay24_x_memory_circuit(H, p)
    sampler = c.compile_sampler(seed=seed)
    ms = sampler.sample(shots)  # shape: (shots, 12 + 24), bits are 0/1

    synd = ms[:, :m].astype(np.uint16)
    data = ms[:, m:].astype(np.uint32)

    # Pack bits into ints (little-endian)
    w12 = (1 << np.arange(m, dtype=np.uint16))
    w24 = (1 << np.arange(n, dtype=np.uint32))

    syn_int = (synd @ w12).astype(np.uint16)      # (shots,)
    data_int = (data @ w24).astype(np.uint32)     # (shots,)

    corr_int = corr_table[syn_int].astype(np.uint32)
    corrected = data_int ^ corr_int               # should be 0 if fully corrected (all-zero codeword)

    fail_bits = (corrected != 0).astype(np.uint8)
    return fail_bits


def decode_outer_fail_bits(H: np.ndarray, corr_table: np.ndarray, err_bits: np.ndarray) -> np.ndarray:
    """
    Outer Golay decoder: given err_bits of shape (shots, 24) (0/1),
    compute syndrome, apply correction, and return fail bits:
      fail=1 if residual != 0 after decoding.
    """
    H = (H.copy() & 1).astype(np.uint8)
    m, n = H.shape
    assert (m, n) == (12, 24)
    assert err_bits.shape[1] == 24
    assert corr_table.size == 4096

    err_bits = err_bits.astype(np.uint8)

    # Syndrome bits and packing
    synd = (err_bits @ H.T) % 2                   # (shots, 12)
    w12 = (1 << np.arange(m, dtype=np.uint16))
    syn_int = (synd.astype(np.uint16) @ w12).astype(np.uint16)

    # Pack error bits into int
    w24 = (1 << np.arange(n, dtype=np.uint32))
    err_int = (err_bits.astype(np.uint32) @ w24).astype(np.uint32)

    corr_int = corr_table[syn_int].astype(np.uint32)
    residual = err_int ^ corr_int

    fail_bits = (residual != 0).astype(np.uint8)
    return fail_bits


# -------------------------
# Main loop: compute p_L for L=1,2,3
# -------------------------
curves = {L: [] for L in levels}

rng = np.random.default_rng(seed_base)

for idx, p in enumerate(p_values):
    # ---- Level 1: true quantum simulation (Stim circuit) ----
    fail1 = sample_level1_fail_bits_from_stim(H, corr_table, float(p), shots_L1, seed=seed_base + 1000 + idx)
    pL1 = float(fail1.mean())
    curves[1].append(pL1)

    # ---- Level 2: sample 24 outputs from level 1 and decode outer ----
    if 2 in levels:
        # Bootstrap: build (shots_L2, 24) by sampling from level-1 outcomes
        err2 = rng.choice(fail1, size=(shots_L2, 24), replace=True)
        fail2 = decode_outer_fail_bits(H, corr_table, err2)
        pL2 = float(fail2.mean())
        curves[2].append(pL2)

    # ---- Level 3: sample 24 outputs from level 2 and decode outer ----
    if 3 in levels:
        err3 = rng.choice(fail2, size=(shots_L3, 24), replace=True)
        fail3 = decode_outer_fail_bits(H, corr_table, err3)
        pL3 = float(fail3.mean())
        curves[3].append(pL3)

    print(f"p={p:.3e} -> L1={pL1:.3e}" +
          (f", L2={pL2:.3e}" if 2 in levels else "") +
          (f", L3={pL3:.3e}" if 3 in levels else ""))

# Convert lists to arrays
for L in levels:
    curves[L] = np.array(curves[L], dtype=float)

# -------------------------
# Plot (log-log)
# -------------------------
plt.figure(figsize=(9, 5))

for L in levels:
    n_phys = 24 ** L
    k_log  = 12 ** L
    label = f"Golay hierarchical L={L} (n={n_phys}, k={k_log}, k/n={k_log/n_phys:.3f})"
    plt.loglog(p_values, np.maximum(curves[L], eps), "o-", label=label)

plt.loglog(p_values, p_values, "--", label="No coding: $p_L = p$")
plt.xlabel("Physical error rate p")
plt.ylabel("Logical / block failure rate $p_L$")
plt.title("Golay(24,12,8): $p_L$ vs $p$ (Stim L1 + hierarchical outer decoding)")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# True L=2 (n=24^2) construction: Golay(24,12,8) product code on a 24x24 grid (576 data qubits).
# Quantum part: Stim circuit generates X errors via X_ERROR(p) and measures all 576 data qubits.
# Classical part: iterative row/column decoding using the Golay syndrome table (corr_table).

import numpy as np
import matplotlib.pyplot as plt
import stim

# ----------------------------
# Parameters (tune as needed)
# ----------------------------
p_values = np.geomspace(3e-3, 2e-1, 18)
shots_L1 = 200_000
shots_L2 = 200_000
num_iters = 2
batch_size = 5_000
num_iters = 2           # number of row/col decoding sweeps
seed_base = 12345
eps = 1e-12

# ----------------------------
# Sanity checks on inputs
# ----------------------------
H = (H.copy() & 1).astype(np.uint8)
assert H.shape == (12, 24)
assert corr_table.size == 4096

# Precompute correction bits for fast XOR:
# corr_bits[s] gives a length-24 bit-vector correction for syndrome s.
corr_bits = ((corr_table[:, None] >> np.arange(24, dtype=np.uint32)) & 1).astype(np.uint8)  # (4096, 24)
w12 = (1 << np.arange(12, dtype=np.uint16))  # weights for packing syndrome bits into an int

def golay_product_L2_circuit(p: float) -> stim.Circuit:
    """
    24x24 data qubits: indices 0..575 mapped as q(r,c)=r*24+c.
    Initialize to |0>, apply X_ERROR(p) to all data qubits, measure all.
    """
    n = 24 * 24
    c = stim.Circuit()
    data = list(range(n))
    c.append("R", data)
    c.append("X_ERROR", data, p)
    c.append("M", data)
    return c

def _decode_rows_inplace(grid: np.ndarray) -> None:
    """
    Decode each of the 24 rows as an independent Golay(24,12,8) block.
    grid shape: (B, 24, 24), uint8 bits.
    """
    B = grid.shape[0]
    rows = grid.reshape(B * 24, 24)                      # (B*24, 24)
    synd = (rows @ H.T) & 1                              # (B*24, 12) over GF(2)
    syn_int = (synd.astype(np.uint16) @ w12).astype(np.uint16)   # (B*24,)
    rows ^= corr_bits[syn_int]                           # apply correction
    grid[:] = rows.reshape(B, 24, 24)

def _decode_cols_inplace(grid: np.ndarray) -> None:
    """
    Decode each of the 24 columns as an independent Golay(24,12,8) block.
    grid shape: (B, 24, 24), uint8 bits.
    """
    B = grid.shape[0]
    gT = grid.transpose(0, 2, 1)                         # (B, 24, 24) now "rows" are original columns
    cols = gT.reshape(B * 24, 24)                        # (B*24, 24)
    synd = (cols @ H.T) & 1                              # (B*24, 12)
    syn_int = (synd.astype(np.uint16) @ w12).astype(np.uint16)
    cols ^= corr_bits[syn_int]
    grid[:] = cols.reshape(B, 24, 24).transpose(0, 2, 1)

def simulate_golay_product_L2_quantum(p: float, shots: int, seed: int = 0, iters: int = 2) -> float:
    """
    Returns block failure rate for the 24x24 Golay×Golay product code.
    Failure if residual grid != 0 after iterative row/column decoding.
    """
    c = golay_product_L2_circuit(p)
    sampler = c.compile_sampler(seed=seed)

    fails = 0
    done = 0

    while done < shots:
        B = min(batch_size, shots - done)
        ms = sampler.sample(B).astype(np.uint8)          # (B, 576) bits
        grid = ms.reshape(B, 24, 24)                     # map into 24x24

        # Iterative product-code decoding: rows then cols (repeat)
        for _ in range(iters):
            _decode_rows_inplace(grid)
            _decode_cols_inplace(grid)

        # Success if all-zero after decoding
        batch_fails = np.any(grid.reshape(B, -1), axis=1).sum()
        fails += int(batch_fails)
        done += B

    return fails / shots

# ----------------------------
# Run: L=1 (optional) + true L=2
# ----------------------------
pL1 = []
pL2 = []

for i, p in enumerate(p_values):
    # L=1 from your existing function (Stim on 24 qubits + Golay decoding)
    ber1 = simulate_golay24_quantum(H, corr_table, float(p), shots_L1, seed=seed_base + 10_000 + i)
    pL1.append(ber1)

    # True L=2 product code on 576 qubits
    ber2 = simulate_golay_product_L2_quantum(float(p), shots_L2, seed=seed_base + 20_000 + i, iters=num_iters)
    pL2.append(ber2)

    print(f"p={p:.3e} -> L1={ber1:.3e},  L2(true)={ber2:.3e}")

pL1 = np.array(pL1, dtype=float)
pL2 = np.array(pL2, dtype=float)

# ----------------------------
# Plot (log-log)
# ----------------------------
plt.figure(figsize=(9, 5))
plt.loglog(p_values, np.maximum(pL1, eps), "o-", label="L=1 Golay(24,12,8) (Stim)")
plt.loglog(p_values, np.maximum(pL2, eps), "s-", label=f"L=2 TRUE (Golay×Golay product, iters={num_iters})")
plt.loglog(p_values, p_values, "--", label="No coding: $p_L=p$")
plt.xlabel("Physical error rate p")
plt.ylabel("Block failure rate $p_L$")
plt.title("Golay: L=1 vs TRUE L=2 (24×24 product code) under X_ERROR(p)")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()